In [1]:
import pandas as pd
import plotly.express as px

In [2]:
raw_df = pd.read_csv("clean_data.csv")

In [3]:
raw_df["order_purchase_timestamp"] = pd.to_datetime(
    raw_df["order_purchase_timestamp"], errors="coerce"
)
raw_df["order_delivered_customer_date"] = pd.to_datetime(
    raw_df["order_delivered_customer_date"], errors="coerce"
)
raw_df["order_estimated_delivery_date"] = pd.to_datetime(
    raw_df["order_estimated_delivery_date"], errors="coerce"
)

raw_df["price"] = pd.to_numeric(raw_df["price"], errors="coerce")
raw_df["freight_value"] = pd.to_numeric(raw_df["freight_value"], errors="coerce")
raw_df["payment_value"] = pd.to_numeric(raw_df["payment_value"], errors="coerce")
raw_df["review_score"] = pd.to_numeric(raw_df["review_score"], errors="coerce")
raw_df["product_weight_g"] = pd.to_numeric(raw_df["product_weight_g"], errors="coerce")


In [4]:
if "product_category_name_english" in raw_df.columns:
    raw_df["category_name"] = raw_df["product_category_name_english"].fillna("Unknown")
else:
    raw_df["category_name"] = raw_df["product_category_name"].fillna("Unknown")

In [6]:
orders_df = raw_df[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "customer_state",
        "review_score",
    ]
].drop_duplicates(subset="order_id")

items_df = raw_df[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "product_weight_g",
        "category_name",
    ]
].drop_duplicates(subset=["order_id", "order_item_id", "product_id", "seller_id"])

payments_df = raw_df[
    [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value",
    ]
].drop_duplicates(subset=["order_id", "payment_sequential"])

In [7]:
item_summary = items_df.groupby("order_id", as_index=False).agg(
    total_items=("order_item_id", "count"),
    price_total=("price", "sum"),
    freight_total=("freight_value", "sum"),
    product_weight_g=("product_weight_g", "mean"),
)

payment_summary = payments_df.groupby("order_id", as_index=False).agg(
    payment_value=("payment_value", "sum")
)

order_df = orders_df.merge(item_summary, on="order_id", how="left")
order_df = order_df.merge(payment_summary, on="order_id", how="left")

order_df["order_value"] = order_df["payment_value"].fillna(
    order_df["price_total"] + order_df["freight_total"]
)
order_df["year_month"] = order_df["order_purchase_timestamp"].dt.to_period("M").astype(str)
order_df["delivery_days"] = (
    order_df["order_delivered_customer_date"] - order_df["order_purchase_timestamp"]
).dt.days



In [8]:
item_summary = items_df.groupby("order_id", as_index=False).agg(
    total_items=("order_item_id", "count"),
    price_total=("price", "sum"),
    freight_total=("freight_value", "sum"),
    product_weight_g=("product_weight_g", "mean"),
)

payment_summary = payments_df.groupby("order_id", as_index=False).agg(
    payment_value=("payment_value", "sum")
)

order_df = orders_df.merge(item_summary, on="order_id", how="left")
order_df = order_df.merge(payment_summary, on="order_id", how="left")

order_df["order_value"] = order_df["payment_value"].fillna(
    order_df["price_total"] + order_df["freight_total"]
)
order_df["year_month"] = order_df["order_purchase_timestamp"].dt.to_period("M").astype(str)
order_df["delivery_days"] = (
    order_df["order_delivered_customer_date"] - order_df["order_purchase_timestamp"]
).dt.days

In [9]:
monthly_revenue = (
    order_df.groupby("year_month", as_index=False)["order_value"]
    .sum()
    .sort_values("year_month")
)

fig1 = px.line(
    monthly_revenue,
    x="year_month",
    y="order_value",
    title="1. Monthly Revenue Trend",
    markers=True,
    color_discrete_sequence=["#ef4444"],
    labels={"year_month": "Month", "order_value": "Revenue (BRL)"},
)
fig1.update_layout(height=380)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'Month=%{x}<br>Revenue (BRL)=%{y}<extra></extra>',
              'legendgroup': '',
              'line': {'color': '#ef4444', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'scatter',
              'x': array(['2016-10', '2016-12', '2017-01', '2017-02', '2017-03', '2017-04',
                          '2017-05', '2017-06', '2017-07', '2017-08', '2017-09', '2017-10',
                          '2017-11', '2017-12', '2018-01', '2018-02', '2018-03', '2018-04',
                          '2018-05', '2018-06', '2018-07', '2018-08'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': ('wvUoXA9Q5kAfhetRuJ4zQHE9CtcDTv' ... 'Uo9CAuQQrXo/BoPixB4XoULstxLUE='),
                    'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'height': 380,
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': '1. Monthly Revenue Trend'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Month'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'Revenue (BRL)'}}}
})

In [10]:
state_revenue = (
    order_df.groupby("customer_state", as_index=False)["order_value"]
    .sum()
    .sort_values("order_value", ascending=False)
)

fig2 = px.bar(
    state_revenue,
    x="order_value",
    y="customer_state",
    orientation="h",
    title="2. Revenue by Customer State",
    color="order_value",
    color_continuous_scale="Viridis",
    labels={"customer_state": "State", "order_value": "Revenue (BRL)"},
)
fig2.update_layout(height=380, coloraxis_showscale=False)


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'Revenue (BRL)=%{marker.color}<br>State=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': {'bdata': ('cT0K12ZQVUHXo3A90DI+QRSuR6EI1T' ... 'MzMxN70kCkcD0Kt2PPQIXrUbg+N8BA'),
                                   'dtype': 'f8'},
                         'coloraxis': 'coloraxis',
                         'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'h',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': ('cT0K12ZQVUHXo3A90DI+QRSuR6EI1T' ... 'MzMxN70kCkcD0Kt2PPQIXrUbg+N8BA'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['SP', 'RJ', 'MG', 'RS', 'PR', 'BA', 'SC', 'DF', 'GO', 'PE', 'ES', 'CE',
                          'PA', 'MT', 'MA', 'MS', 'PB', 'PI', 'RN', 'AL', 'SE', 'TO', 'RO', 'AM',
                          'AC', 'AP', 'RR'], dtype=object),
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'coloraxis': {'colorbar': {'title': {'text': 'Revenue (BRL)'}},
                             'colorscale': [[0.0, '#440154'], [0.1111111111111111,
                                            '#482878'], [0.2222222222222222,
                                            '#3e4989'], [0.3333333333333333,
                                            '#31688e'], [0.4444444444444444,
                                            '#26828e'], [0.5555555555555556,
                                            '#1f9e89'], [0.6666666666666666,
                                            '#35b779'], [0.7777777777777778,
                                            '#6ece58'], [0.8888888888888888,
                                            '#b5de2b'], [1.0, '#fde725']],
                             'showscale': False},
               'height': 380,
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': '2. Revenue by Customer State'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Revenue (BRL)'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'State'}}}
})

In [11]:
top_categories = (
    items_df.groupby("category_name", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "revenue"})
    .sort_values("revenue", ascending=False)
    .head(10)
)

fig3 = px.bar(
    top_categories,
    x="revenue",
    y="category_name",
    orientation="h",
    title="3. Top 10 Product Categories",
    color="revenue",
    color_continuous_scale="Plasma",
    labels={"category_name": "Category", "revenue": "Revenue (BRL)"},
)
fig3.update_layout(height=380, coloraxis_showscale=False)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'Revenue (BRL)=%{marker.color}<br>Category=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': {'bdata': ('mpmZGQdcMkH2KFzPHXcxQTMzMzNSdy' ... '9C2EIiQdejcL20QiFBXI/C9YkuHEE='),
                                   'dtype': 'f8'},
                         'coloraxis': 'coloraxis',
                         'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'h',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': ('mpmZGQdcMkH2KFzPHXcxQTMzMzNSdy' ... '9C2EIiQdejcL20QiFBXI/C9YkuHEE='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['health_beauty', 'watches_gifts', 'bed_bath_table', 'sports_leisure',
                          'computers_accessories', 'furniture_decor', 'housewares', 'cool_stuff',
                          'auto', 'garden_tools'], dtype=object),
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'coloraxis': {'colorbar': {'title': {'text': 'Revenue (BRL)'}},
                             'colorscale': [[0.0, '#0d0887'], [0.1111111111111111,
                                            '#46039f'], [0.2222222222222222,
                                            '#7201a8'], [0.3333333333333333,
                                            '#9c179e'], [0.4444444444444444,
                                            '#bd3786'], [0.5555555555555556,
                                            '#d8576b'], [0.6666666666666666,
                                            '#ed7953'], [0.7777777777777778,
                                            '#fb9f3a'], [0.8888888888888888,
                                            '#fdca26'], [1.0, '#f0f921']],
                             'showscale': False},
               'height': 380,
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': '3. Top 10 Product Categories'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Revenue (BRL)'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'Category'}}}
})

In [12]:
payment_mix = (
    payments_df.groupby("payment_type", as_index=False)["payment_value"]
    .sum()
    .sort_values("payment_value", ascending=False)
)

fig4 = px.pie(
    payment_mix,
    names="payment_type",
    values="payment_value",
    title="4. Payment Method Distribution",
    hole=0.35,
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig4.update_layout(height=380)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
              'hole': 0.35,
              'hovertemplate': 'payment_type=%{label}<br>payment_value=%{value}<extra></extra>',
              'labels': array(['credit_card', 'boleto', 'voucher', 'debit_card'], dtype=object),
              'legendgroup': '',
              'name': '',
              'showlegend': True,
              'type': 'pie',
              'values': {'bdata': 'PQrXc5xOZkGF61GYi1lEQWZmZmZXEBRBSOF6FNYqCEE=', 'dtype': 'f8'}}],
    'layout': {'height': 380,
               'legend': {'tracegroupgap': 0},
               'piecolorway': [rgb(127, 60, 141), rgb(17, 165, 121), rgb(57, 105,
                               172), rgb(242, 183, 1), rgb(231, 63, 116), rgb(128,
                               186, 90), rgb(230, 131, 16), rgb(0, 134, 149),
                               rgb(207, 28, 144), rgb(249, 123, 114), rgb(165, 170,
                               153)],
               'template': '...',
               'title': {'text': '4. Payment Method Distribution'}}
})

In [13]:
delivery_data = order_df["delivery_days"].dropna()
delivery_data = delivery_data[delivery_data <= 60]

fig5 = px.histogram(
    x=delivery_data,
    nbins=30,
    title="5. Delivery Time Distribution",
    labels={"x": "Delivery Time (Days)", "y": "Number of Orders"},
    color_discrete_sequence=["#06b6d4"],
)
fig5.update_layout(height=380)


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'Delivery Time (Days)=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#06b6d4', 'pattern': {'shape': ''}},
              'name': '',
              'nbinsx': 30,
              'orientation': 'v',
              'showlegend': False,
              'type': 'histogram',
              'x': {'bdata': ('CA0JDQIQCQkSDAUMBAsMBg0VBAQSBB' ... '0EDA4KJwoUBwYKAggOJRALCwgWGBEH'),
                    'dtype': 'i1'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'height': 380,
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': '5. Delivery Time Distribution'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Delivery Time (Days)'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

In [14]:
correlation_data = order_df[
    ["price_total", "freight_total", "payment_value", "product_weight_g", "review_score"]
].corr(numeric_only=True)

correlation_data.index = ["Price", "Freight", "Payment", "Weight", "Review"]
correlation_data.columns = ["Price", "Freight", "Payment", "Weight", "Review"]

fig6 = px.imshow(
    correlation_data,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="6. Correlation Heatmap",
)
fig6.update_layout(height=380)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<br>y: %{y}<br>color: %{z}<extra></extra>',
              'name': '0',
              'texttemplate': '%{z:.2f}',
              'type': 'heatmap',
              'x': array(['Price', 'Freight', 'Payment', 'Weight', 'Review'], dtype=object),
              'xaxis': 'x',
              'y': array(['Price', 'Freight', 'Payment', 'Weight', 'Review'], dtype=object),
              'yaxis': 'y',
              'z': {'bdata': ('AAAAAAAA8D9bvgicHCraPxsciDPc3O' ... '21GXalv0oGI6WA7J2/AAAAAAAA8D8='),
                    'dtype': 'f8',
                    'shape': '5, 5'}}],
    'layout': {'coloraxis': {'cmax': 1,
                             'cmin': -1,
                             'colorscale': [[0.0, 'rgb(5,48,97)'], [0.1,
                                            'rgb(33,102,172)'], [0.2,
                                            'rgb(67,147,195)'], [0.3,
                                            'rgb(146,197,222)'], [0.4,
                                            'rgb(209,229,240)'], [0.5,
                                            'rgb(247,247,247)'], [0.6,
                                            'rgb(253,219,199)'], [0.7,
                                            'rgb(244,165,130)'], [0.8,
                                            'rgb(214,96,77)'], [0.9,
                                            'rgb(178,24,43)'], [1.0,
                                            'rgb(103,0,31)']]},
               'height': 380,
               'template': '...',
               'title': {'text': '6. Correlation Heatmap'},
               'xaxis': {'anchor': 'y', 'constrain': 'domain', 'domain': [0.0, 1.0], 'scaleanchor': 'y'},
               'yaxis': {'anchor': 'x', 'autorange': 'reversed', 'constrain': 'domain', 'domain': [0.0, 1.0]}}
})